In [1]:
#pip install nltk
#pip install pandas
#pip install numpy
#pip install scipy
#pip install matplotlib
#pip install textblob
#pip install scikit-learn



In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from textblob import TextBlob
import nltk
import numpy as np
import scipy
import sklearn


# Importing the Data

In [3]:
news_data = pd.read_csv("C:/Users/MATie/OneDrive/Documents/CSE 6242/Project/processed_combined_data.csv")
# news_data = news_data[["label", "author", "text", "date", "dataset", "type", "title", "subject", "extracted_source"]]

C:\Users\MATie\AppData\Local\Temp\ipykernel_60188\3555750658.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  news_data = pd.read_csv("C:/Users/MATie/OneDrive/Documents/CSE 6242/Project/processed_combined_data.csv")


## Data Cleaning

In [4]:
news_data[["Unnamed: 0", "label", "author", "text", "date", "type", "title", "subject", "extracted_source"]].head(1)
news_data = news_data[["Unnamed: 0", "label", "author", "text", "date", "type", "title", "subject", "extracted_source"]]

news_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 116038 entries, 0 to 116037
Data columns (total 9 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   Unnamed: 0        116038 non-null  int64 
 1   label             116038 non-null  int64 
 2   author            21152 non-null   object
 3   text              115904 non-null  object
 4   date              33004 non-null   object
 5   type              83546 non-null   object
 6   title             85453 non-null   object
 7   subject           54278 non-null   object
 8   extracted_source  36839 non-null   object
dtypes: int64(2), object(7)
memory usage: 8.0+ MB


Ensuring labels are 0 through 4 and that there are no rows without a label.

In [5]:
news_data["label"].unique()

array([4, 0, 3, 2, 1])

In [6]:
news_data[news_data["label"].isna()]

,Unnamed: 0,label,author,text,date,type,title,subject,extracted_source


Clearing out rows with empty text cells.


In [7]:
news_data = news_data[news_data["text"].ne(" ") & news_data['text'].notna() & news_data['text'].ne("") & news_data['text'].ne("  ")]

Converting text, title, and subject columns to strings. Change the label to an integer.

In [8]:
news_data["label"] = news_data["label"].astype("int")
news_data["text"] = news_data["text"].astype(str)
news_data["title"] = news_data["title"].astype(str)
news_data["subject"] = news_data["subject"].astype(str)

### Getting rid of pesky smart quotes and other strange encodings:

In [9]:
# removing strange characters
def clean_encoding(text):
    text = text = text.replace('â€˜', "'").replace('â€™', "'").replace('â€œ', '"').replace('â€', '"')
    text = text.replace('â€“', '–').replace('â€”', '—').replace('â€', '')
    return text

In [10]:
news_data["text"] = news_data["text"].apply(clean_encoding)

## Getting Word Count

In [11]:
def get_article_word_count(article):
    return len(article.split())

In [12]:
news_data["word_count"] = news_data["text"].apply(get_article_word_count)
news_data.head(3)

,Unnamed: 0,label,author,text,date,type,title,subject,extracted_source,word_count
0,0,4,Barack Obama,John McCain opposed bankruptcy protections for...,6/11/2008,statement,nan,nan,NaN,19
1,1,0,Matt Gaetz,Bennie Thompson actively cheer-led riots in th...,6/7/2022,statement,nan,nan,NaN,8
2,2,3,Kelly Ayotte,Says Maggie Hassan was out of state on 30 days...,5/18/2016,statement,nan,nan,NaN,15


## Polarity and Subjectivity Scores for Headlines and Text via TextBlob

TextBlob breaks text sentiment down to polarity (how positive or negative a text is) and subjectivity (how objective or subjective a text is).

I add columns for overall text polarity and subjectivity, then lastly calculate the average polarity and subjectivity by sentence (just to see if a fine-grain calculation would show a different score, since a positive+negative word negates to 0. I was curious how this might be effected at the individual level). 

Supposedly, fake news headlines have specific styles, e.g. use more words than real articles to be more attention grabbing. Since our dataset containers titleless statements, headline polarity wasn't calculated, but potentially would be a very good feature to have.

References for TextBlob usage:

1) https://www.youtube.com/watch?app=desktop&v=1zk_leGDzqI&t=702s

2) For how TextBlob works:  https://planspace.org/20150607-textblob_sentiment/

3) https://textblob.readthedocs.io/en/dev/advanced_usage.html#sentiment-analyzers

In [13]:
# polarity score is a float within the range [-1.0, 1.0]
# -1 is very negative; 1 is very positive.
# subjectivity score is a float within the range [0.0, 1.0]
# 0 is very objective; 1 is very subjective.

def find_polarity(news_story):
    news = TextBlob(news_story)
    return news.sentiment.polarity

def find_subjectivity(news_story):
    news = TextBlob(news_story)
    return news.sentiment.subjectivity

    

In [14]:
news_data['Polarity'] = news_data['text'].apply(find_polarity)
news_data['Subjectivity'] = news_data['text'].apply(find_subjectivity)


In [15]:
news_data.head(3)

,Unnamed: 0,label,author,text,date,type,title,subject,extracted_source,word_count,Polarity,Subjectivity
0,0,4,Barack Obama,John McCain opposed bankruptcy protections for...,6/11/2008,statement,nan,nan,NaN,19,0.000000,0.500000
1,1,0,Matt Gaetz,Bennie Thompson actively cheer-led riots in th...,6/7/2022,statement,nan,nan,NaN,8,-0.133333,0.600000
2,2,3,Kelly Ayotte,Says Maggie Hassan was out of state on 30 days...,5/18/2016,statement,nan,nan,NaN,15,0.000000,0.066667


# Getting the sentiment/subjectivity scores using sentence averages 

In [16]:
# could probably improve speed using numpy

def find_avg_polarity_sentences(news_story):
    news = TextBlob(news_story)
    polarity = []
    for sentence in news.sentences:
        polarity.append(sentence.sentiment.polarity)
    return sum(polarity) / len(polarity)

def find_avg_subjectivity_sentences(news_story):
    news = TextBlob(news_story)
    subjectivity = []
    for sentence in news.sentences:
        subjectivity.append(sentence.sentiment.subjectivity)
    return sum(subjectivity) / len(subjectivity)

In [17]:
text_list = news_data['text'].tolist()

for text in text_list:
    try:
        find_avg_polarity_sentences(text)
    except:
        print(text_list.index(text))
        continue

In [18]:
text_list = news_data['text'].tolist()

avg_polarity = []
avg_subjectivity = []

for text in text_list:
    avg_polarity.append(find_avg_polarity_sentences(text))
    avg_subjectivity.append(find_avg_subjectivity_sentences(text))



In [19]:
news_data["avg_polarity"] = avg_polarity
news_data["avg_subjectivity"] = avg_subjectivity

In [20]:
news_data.head(3)

,Unnamed: 0,label,author,text,date,type,title,subject,extracted_source,word_count,Polarity,Subjectivity,avg_polarity,avg_subjectivity
0,0,4,Barack Obama,John McCain opposed bankruptcy protections for...,6/11/2008,statement,nan,nan,NaN,19,0.000000,0.500000,0.000000,0.500000
1,1,0,Matt Gaetz,Bennie Thompson actively cheer-led riots in th...,6/7/2022,statement,nan,nan,NaN,8,-0.133333,0.600000,-0.133333,0.600000
2,2,3,Kelly Ayotte,Says Maggie Hassan was out of state on 30 days...,5/18/2016,statement,nan,nan,NaN,15,0.000000,0.066667,0.000000,0.066667


## Getting the Topics/Subjects Using NMF

In [21]:
# Topic Identification
# https://www.analyticsvidhya.com/blog/2022/02/topic-identification-with-gensim-library-using-python/
# https://www.youtube.com/watch?v=OYze4BQtn-U
#https://medium.com/blend360/topic-modelling-a-comparison-between-lda-nmf-bertopic-and-top2vec-part-i-3c16372d51f0
# last one has good refs for lit review!

# https://www.youtube.com/watch?v=_QiTQQDrx5I

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn import decomposition
nltk.download("stopwords")
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\MATie\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [23]:
stopword_list = stopwords.words('english')

stopword_list.extend(["reuters", "monday", "tuesday", "wednesday", "thursday", "friday", "2016", "2017", "2018", "2019", "2020"])


In [24]:
#https://scikit-learn.org/stable/auto_examples/applications/plot_topics_extraction_with_nmf_lda.html
# Use tf-idf features for NMF.
print("Extracting tf-idf features for NMF...")
tfidf_vectorizer = TfidfVectorizer(
    stop_words = stopword_list, min_df = 20, max_df = 0.95, ngram_range = (1,2), strip_accents = 'unicode', max_features = 1000) 


Extracting tf-idf features for NMF...


In [25]:
tfidf = tfidf_vectorizer.fit_transform(text_list)



In [26]:
tfidf_vectorizer.get_feature_names_out()

array(['000', '10', '100', '11', '12', '13', '14', '15', '16', '17', '18',
       '20', '2012', '2013', '2014', '2015', '24', '25', '30', '50',
       'able', 'abortion', 'absolutely', 'access', 'according', 'account',
       'accused', 'across', 'act', 'action', 'actions', 'actually',
       'added', 'adding', 'address', 'administration', 'adviser',
       'african', 'agencies', 'agency', 'agenda', 'ago', 'agreed',
       'agreement', 'ahead', 'aid', 'air', 'al', 'allegations', 'alleged',
       'allies', 'allow', 'allowed', 'almost', 'along', 'already', 'also',
       'although', 'always', 'ambassador', 'america', 'american',
       'americans', 'among', 'announced', 'another', 'anti', 'anyone',
       'anything', 'apparently', 'appeared', 'april', 'arabia', 'area',
       'areas', 'armed', 'army', 'around', 'arrested', 'article', 'ask',
       'asked', 'attack', 'attacks', 'attempt', 'attorney',
       'attorney general', 'august', 'authorities', 'away', 'back',
       'backed', 'ba

In [27]:
num_topics = 20
model = decomposition.NMF(n_components=num_topics, random_state = 42) 
W = model.fit_transform(tfidf)
H = model.components_

In [28]:
# coefficient matrix
H

array([[0.00000000e+00, 9.64603653e-02, 1.34160342e-01, ...,
        6.15554705e-01, 0.00000000e+00, 8.18401466e-01],
       [0.00000000e+00, 0.00000000e+00, 1.70761428e-02, ...,
        4.32692193e-02, 2.03720287e-01, 0.00000000e+00],
       [3.93521666e+00, 8.91691515e-01, 6.30317903e-01, ...,
        7.21759541e-02, 7.62893772e-01, 2.47632573e-02],
       ...,
       [0.00000000e+00, 1.07825695e-03, 4.59023848e-03, ...,
        1.67361438e-02, 2.22646561e-02, 0.00000000e+00],
       [0.00000000e+00, 5.85926213e-03, 0.00000000e+00, ...,
        5.76523467e-02, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 1.35037321e-01, 1.83681797e-02, ...,
        1.47904906e-02, 0.00000000e+00, 1.34191564e-02]], shape=(20, 1000))

In [29]:
# dictionary matrix

W

array([[0.00068626, 0.        , 0.00309755, ..., 0.        , 0.        ,
        0.00062546],
       [0.        , 0.        , 0.        , ..., 0.        , 0.00132128,
        0.        ],
       [0.        , 0.        , 0.01236723, ..., 0.        , 0.        ,
        0.00147327],
       ...,
       [0.01254927, 0.03798719, 0.        , ..., 0.00300248, 0.        ,
        0.01582571],
       [0.        , 0.        , 0.00576968, ..., 0.        , 0.        ,
        0.00064143],
       [0.00612988, 0.00461778, 0.        , ..., 0.00036342, 0.        ,
        0.00551669]], shape=(115268, 20))

In [30]:
# extract significant words in each topic

num_words = 15

# get feature names as array
vocab = np.array(tfidf_vectorizer.get_feature_names_out())
# higher coefficients towards end of array, pick last 15 words
top_words = lambda t: [vocab[i] for i in np.argsort(t)[:-num_words-1:-1]]
# get top words from H matrix
topic_words = ([top_words(t) for t in H])
topics = [" ".join(t) for t in topic_words]


# build topic and word dataframe

In [31]:
topics

['people like one women right know us get even america via time video going image',
 'trump donald donald trump president campaign trump said republican president trump supporters presidential image trump campaign images trumps featured',
 '000 million billion state year jobs budget new federal money years government spending city debt',
 'health care insurance obamacare bill healthcare law plan act americans medical people government reform program',
 'clinton hillary hillary clinton campaign sanders democratic emails email clintons presidential fbi election foundation state candidate',
 'says voted texas joe taxes wants shows donald donald trump cut abortion photo county century school',
 'korea north north korea nuclear china korean south missile sanctions japan chinese beijing weapons military tillerson',
 'obama president barack barack obama president obama president barack administration obama administration obamas bush office first american americans former',
 'said would statem

## Extracting how many times a speech tag is included in the text


In [32]:
# get number of speech attributes

def get_num_speech_attributes(text):
    text = text.lower()
    return text.count("said") + text.count("say") + text.count("told") + text.count("tell") 

In [33]:
get_num_speech_attributes(news_data["text"][2])

1

In [34]:
news_data["num_speech_tags"] = news_data["text"].apply(get_num_speech_attributes)

In [35]:
news_data.head(3)

,Unnamed: 0,label,author,text,date,type,title,subject,extracted_source,word_count,Polarity,Subjectivity,avg_polarity,avg_subjectivity,num_speech_tags
0,0,4,Barack Obama,John McCain opposed bankruptcy protections for...,6/11/2008,statement,nan,nan,NaN,19,0.000000,0.500000,0.000000,0.500000,0
1,1,0,Matt Gaetz,Bennie Thompson actively cheer-led riots in th...,6/7/2022,statement,nan,nan,NaN,8,-0.133333,0.600000,-0.133333,0.600000,0
2,2,3,Kelly Ayotte,Says Maggie Hassan was out of state on 30 days...,5/18/2016,statement,nan,nan,NaN,15,0.000000,0.066667,0.000000,0.066667,1


## COMBINING DATAFRAMES

In [36]:
molly_data = news_data[["Unnamed: 0", "label", "text", "type", "word_count", "Polarity", "Subjectivity", "avg_polarity", "avg_subjectivity", "num_speech_tags"]]

molly_data.head(1)

,Unnamed: 0,label,text,type,word_count,Polarity,Subjectivity,avg_polarity,avg_subjectivity,num_speech_tags
0,0,4,John McCain opposed bankruptcy protections for...,statement,19,0.0,0.5,0.0,0.5,0


In [37]:
dylan_data = pd.read_csv("C:/Users/MATie/OneDrive/Documents/CSE 6242/Project/dylan_data.csv")
dylan_data.head(1)

C:\Users\MATie\AppData\Local\Temp\ipykernel_60188\1612443389.py:1: DtypeWarning: Columns (0,1,1004) have mixed types. Specify dtype option on import or set low_memory=False.
  dylan_data = pd.read_csv("C:/Users/MATie/OneDrive/Documents/CSE 6242/Project/dylan_data.csv")


,Unnamed: 0,label,text,cleaned_text,able,absolutely,access,according,account,accused,...,wrote,xi,year,yearold,years,yes,york,young,label.1,flesch_reading_ease
0,9745,2,"When a bill is sent to the governors office, t...",when a bill is sent to the governors office th...,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,74.9


In [38]:
combined_df = pd.merge(molly_data, dylan_data, on = "Unnamed: 0", how = "inner")

In [39]:
combined_df['label_x'].equals(combined_df['label_y'])

False

In [40]:
different_rows = combined_df[combined_df['label_x'] != combined_df['label_y']]
different_rows

,Unnamed: 0,label_x,text_x,type,word_count,Polarity,Subjectivity,avg_polarity,avg_subjectivity,num_speech_tags,...,wrote,xi,year,yearold,years,yes,york,young,label.1,flesch_reading_ease


In [41]:
combined_df.drop("label_y", inplace = True, axis = 1)

In [42]:
combined_df['text_x'].equals(combined_df['text_y'])

True

In [43]:
combined_df.drop(["text_y", "label.1"], inplace = True, axis = 1)

In [44]:
combined_df.rename(columns = {"label_x": "label", "text_x": "text"}, inplace = True)

In [45]:
combined_df.head(1)

,Unnamed: 0,label,text,type,word_count,Polarity,Subjectivity,avg_polarity,avg_subjectivity,num_speech_tags,...,wrong,wrote,xi,year,yearold,years,yes,york,young,flesch_reading_ease
0,2,3,Says Maggie Hassan was out of state on 30 days...,statement,15,0.0,0.066667,0.0,0.066667,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,90.09


In [46]:
combined_df.rename(columns={"Unnamed: 0": "index", "Subjectivity": "subjectivity", "Polarity": "polarity"}, inplace = True)
combined_df.to_csv(r"C:/Users/MATie/OneDrive/Documents/CSE 6242/Project/combined_data.csv", index = False)


## Multinomial Regression

In [47]:
from sklearn.model_selection import train_test_split

In [49]:
#https://www.bing.com/videos/riverview/relatedvideo?q=scikit+learn+multinomial+regression&mid=7FAB78DABA82344585B97FAB78DABA82344585B9&FORM=VIRE

X = combined_df.drop(["index", "label", "text", "cleaned_text", "type"], axis = 1)
y = combined_df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [50]:
X.head(1)

,word_count,polarity,subjectivity,avg_polarity,avg_subjectivity,num_speech_tags,able,absolutely,access,according,...,wrong,wrote,xi,year,yearold,years,yes,york,young,flesch_reading_ease
0,15,0.0,0.066667,0.0,0.066667,1,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,90.09


In [51]:
#https://scikit-learn.org/stable/modules/preprocessing.html
from sklearn import preprocessing
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
pipe = make_pipeline(StandardScaler(), LogisticRegression(random_state = 42))
pipe.fit(X_train, y_train)  # apply scaling on training data

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression', LogisticRegression(random_state=42))])

In [52]:
pipe.score(X_test, y_test)

0.7303921568627451

In [53]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred = pipe.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

[[218   7   6  11  28]
 [  5   3   8   4   8]
 [ 14   2   6   6   8]
 [ 10   3   9   5  10]
 [ 15   2   2   7 215]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.81      0.82       270
           1       0.18      0.11      0.13        28
           2       0.19      0.17      0.18        36
           3       0.15      0.14      0.14        37
           4       0.80      0.89      0.84       241

    accuracy                           0.73       612
   macro avg       0.43      0.42      0.42       612
weighted avg       0.71      0.73      0.72       612



In [54]:
print(pipe.named_steps['logisticregression'].classes_)
pipe.named_steps['logisticregression'].coef_

[0 1 2 3 4]


array([[ 0.26846179, -0.13299457,  0.19158099, ..., -0.01352331,
        -0.03012092, -0.00570697],
       [-0.04837668,  0.02143134, -0.02959177, ..., -0.09526215,
         0.00851973,  0.27537428],
       [-0.23582032,  0.14107951, -0.17105225, ...,  0.01045704,
        -0.07583746,  0.0730051 ],
       [-0.16831482, -0.08342536,  0.11388357, ...,  0.09256065,
         0.01872274, -0.29002882],
       [ 0.18405003,  0.05390908, -0.10482053, ...,  0.00576777,
         0.0787159 , -0.05264359]], shape=(5, 1007))

In [55]:
sklearn.svm.LinearSVC(multi_class = "crammer_singer").fit(X_train, y_train).score(X_test, y_test)

c:\Users\MATie\CSE 6242\TruthFinder\venv\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


0.7205882352941176

In [56]:
sklearn.tree.DecisionTreeClassifier().fit(X_train, y_train).score(X_test, y_test)

0.7679738562091504

In [57]:
sklearn.neighbors.KNeighborsClassifier().fit(X_train, y_train).score(X_test, y_test)

0.6241830065359477

In [58]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC
OneVsRestClassifier(SVC()).fit(X_train, y_train).score(X_test, y_test)

0.5147058823529411

In [59]:
from sklearn.metrics import accuracy_score

In [60]:
import xgboost as xgb

num_classes = len(set(y))  # Calculate the number of unique classes in y
model = xgb.XGBClassifier(objective='multi:softmax', num_class=num_classes, n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)
model.fit(X_train, y_train)
 
# Step 3: Make predictions
y_pred = model.predict(X_test)
 
# Step 4: Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.7908496732026143
Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.95      0.84       270
           1       0.11      0.04      0.05        28
           2       0.19      0.11      0.14        36
           3       0.36      0.14      0.20        37
           4       0.96      0.90      0.93       241

    accuracy                           0.79       612
   macro avg       0.47      0.43      0.43       612
weighted avg       0.75      0.79      0.76       612



# Can we get better performance if we split the dataset by statement/article?

In [62]:
molly_data2 = news_data[["Unnamed: 0", "label", "text", "type", "title", "word_count", "Polarity", "Subjectivity", "avg_polarity", "avg_subjectivity", "num_speech_tags"]]
dylan_data2 = pd.read_csv("C:/Users/MATie/OneDrive/Documents/CSE 6242/Project/dylan_data.csv")
combined_df2 = pd.merge(molly_data2, dylan_data2, on = "Unnamed: 0", how = "inner")
combined_df2.drop(["label_y", "text_y", "label.1"], inplace = True, axis = 1)
combined_df2.rename(columns = {"label_x": "label", "text_x": "text"}, inplace = True)


C:\Users\MATie\AppData\Local\Temp\ipykernel_60188\1752808558.py:2: DtypeWarning: Columns (0,1,1004) have mixed types. Specify dtype option on import or set low_memory=False.
  dylan_data2 = pd.read_csv("C:/Users/MATie/OneDrive/Documents/CSE 6242/Project/dylan_data.csv")


In [63]:
#combined_df2.head(3)
combined_df2.to_csv(r"C:/Users/MATie/OneDrive/Documents/CSE 6242/Project/combined_data2.csv", index = False)

In [64]:
combined_df2["type"].unique()
combined_df2[combined_df2["type"].isna() & combined_df2["title"].isna()]

,Unnamed: 0,label,text,type,title,word_count,Polarity,Subjectivity,avg_polarity,avg_subjectivity,...,wrong,wrote,xi,year,yearold,years,yes,york,young,flesch_reading_ease


Because every type missing a type attribute has a title, we can assume that the text is an article, and not a statement.

In [65]:
combined_df2["type"] = combined_df2["type"].fillna("article")

Statements = 1, Articles = 0

In [66]:
combined_df2["type"].unique()
combined_df2['type'] = combined_df2['type'].replace({'statement': 1, 'article': 0})

C:\Users\MATie\AppData\Local\Temp\ipykernel_60188\625182345.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  combined_df2['type'] = combined_df2['type'].replace({'statement': 1, 'article': 0})


## Splitting the Dataframe by Text Type

In [67]:
statements = combined_df2[combined_df2["type"] == 1]
articles = combined_df2[combined_df2["type"] == 0]

In [68]:
articles.head(1)

,Unnamed: 0,label,text,type,title,word_count,Polarity,Subjectivity,avg_polarity,avg_subjectivity,...,wrong,wrote,xi,year,yearold,years,yes,york,young,flesch_reading_ease
566,21170,0,Abigail Disney is an heiress with brass ovarie...,0,Heiress To Disney Empire Knows GOP Scammed Us...,500,0.098287,0.531089,0.034911,0.30447,...,0.0,0.122134,0.0,0.0,0.0,0.0,0.0,0.0,0.0,64.51


## Testing Models

In [69]:
X_art = articles.drop(["Unnamed: 0", "text", "title", "cleaned_text" ], axis = 1)
#X_art.info()
y_art = articles["label"]
X_train, X_test, y_train, y_test = train_test_split(X_art, y_art, test_size = 0.2, random_state = 42)

In [70]:
pipe = make_pipeline(StandardScaler(), LogisticRegression(random_state = 42))
pipe.fit(X_train, y_train)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression', LogisticRegression(random_state=42))])

In [71]:
y_pred = pipe.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

[[213   2]
 [  2 239]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       215
           4       0.99      0.99      0.99       241

    accuracy                           0.99       456
   macro avg       0.99      0.99      0.99       456
weighted avg       0.99      0.99      0.99       456



In [72]:
pipe.score(X_test, y_test)

0.9912280701754386

In [73]:
X_state = statements.drop(["Unnamed: 0", "text", "title", "cleaned_text" ], axis = 1)
#X_art.info()
y_state = statements["label"]
X_train, X_test, y_train, y_test = train_test_split(X_state, y_state, test_size = 0.2, random_state = 42)

In [74]:
pipe = make_pipeline(StandardScaler(), LogisticRegression(random_state = 42))
pipe.fit(X_train, y_train)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression', LogisticRegression(random_state=42))])

In [75]:
y_pred = pipe.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

[[29  7  4  4  0]
 [11  9  5  1  1]
 [12  8  7  9  2]
 [ 1  4  8  9  5]
 [ 0  2  4  6  8]]

Classification Report:
              precision    recall  f1-score   support

           0       0.55      0.66      0.60        44
           1       0.30      0.33      0.32        27
           2       0.25      0.18      0.21        38
           3       0.31      0.33      0.32        27
           4       0.50      0.40      0.44        20

    accuracy                           0.40       156
   macro avg       0.38      0.38      0.38       156
weighted avg       0.38      0.40      0.39       156



In [76]:
pipe.score(X_test, y_test)

0.3974358974358974

In [77]:
news_data.shape

(115268, 15)

In [78]:
dylan_data.shape

(4993, 1006)